In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
import json
import time
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from dataset import create_dataloaders

# LSTM class definition
class FallDetectionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(FallDetectionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 2)  # 2 classes: fall, no-fall

    # LSTM forward pass
    def forward(self, x):
        batch_size = x.size(0)

        # Flatten keypoints: [batch_size, 60, 17, 3] -> [batch_size, 60, 51] 
        # Each frame becomes: [x, y, confidence, x, y, confidence, ..]
        x = x.view(batch_size, 60, -1)

        # Process Batch
        # lstm_out: outputs from last layer
        # h_n, c_n: final hidden states and cell states for each sequence in each layer [num_layers, batch_size, hidden_size] 
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Use the final hidden states for each sequence from the final layer
        # h_n[-1]: [batch_size, hidden_size]
        last_hidden = h_n[-1]

        # Pass final hidden states into fully connected layers 
        out = self.fc1(last_hidden)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)

        return out

# Define early stopping
class EarlyStopping:
    def __init__(self, patience, min_delta):
        self.patience = patience 
        self.min_delta = min_delta 
        self.counter = 0
        self.best_f1 = None
        self.early_stop = False

    # If the model's f1 has not improved in the specified number of epochs, stop training
    def __call__(self, val_f1):
        if self.best_f1 is None:
            self.best_f1 = val_f1
        elif val_f1 < self.best_f1 + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_f1 = val_f1
            self.counter = 0


def train_one_epoch(model, train_loader, criterion, optimiser, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    # Iterate through batches in train loader
    for batch_idx, batch in enumerate(train_loader):
        # Move data to device
        keypoints = batch['keypoints'].to(device)  # [batch_size, 60, 17, 3]
        labels = batch['label'].to(device) 

        # Reset gradients
        optimiser.zero_grad()

        # Forward pass
        outputs = model(keypoints)  # [batch_size, 2]
        loss = criterion(outputs, labels)

        # Backward pass calculates gradients
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Progress
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    # Calculate metrics
    epoch_loss = running_loss / len(train_loader)
    accuracy = accuracy_score(all_labels, all_preds)

    return epoch_loss, accuracy

# Validation between epochs to identify model improvement
def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        # Iterate through batches
        for batch in val_loader:
            keypoints = batch['keypoints'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            outputs = model(keypoints)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    val_loss = running_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )

    return val_loss, accuracy, precision, recall, f1

# Final evaluation on validation set
def evaluate(model, val_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        # Iterate through batches
        for batch in val_loader:
            keypoints = batch['keypoints'].to(device)
            labels = batch['label'].to(device)
            
            # Forward pass
            outputs = model(keypoints)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds)

    return accuracy, precision, recall, f1, cm, all_preds, all_labels


def train_model(model, train_loader, val_loader, num_epochs, patience, min_delta, learning_rate, 
                weight_decay, class_weights, save_dir):
    
    # Use GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")

    model = model.to(device)

    # Weighted loss function (penalise missing falls more)
    class_weights_tensor = torch.FloatTensor(class_weights).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) # Define criterion
    print(f"Using class weights: no-fall={class_weights[0]}, fall={class_weights[1]}")

    # Optimiser with L2 regularisation (weight_decay)
    # L2 regularisation adds a penalty to parameters to prevent them from getting too large, reducing overfitting
    optimiser = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Learning rate scheduler
    # Reduces learning rate when a metric stops improving
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode='min', factor=0.5, patience=5, verbose=True
    )

    # Early stopping 
    early_stopping = EarlyStopping(patience, min_delta)

    # Create save directory
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': []
    }

    best_f1 = 0.0
    best_epoch = 0

    print("\n" + "=" * 70)
    print("TRAINING LSTM MODEL - v11")
    print("=" * 70)

    for epoch in range(num_epochs):
        epoch_start = time.time()

        print(f"\nEpoch [{epoch+1}/{num_epochs}]")

        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimiser, device
        )

        # Validate
        val_loss, val_acc, val_precision, val_recall, val_f1 = validate(
            model, val_loader, criterion, device
        )

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_precision'].append(val_precision)
        history['val_recall'].append(val_recall)
        history['val_f1'].append(val_f1)

        # Print epoch summary
        epoch_time = time.time() - epoch_start
        print(f"\n  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print(f"  Val Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
        print(f"  Time: {epoch_time:.1f}s")

        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimiser.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
            }, save_path / 'best_model.pth')
            print(f"  ! Saved best model (F1: {best_f1:.4f})")

        # Early stopping check
        early_stopping(val_f1)
        if early_stopping.early_stop:
            print(f"\n  Early stopping triggered at epoch {epoch+1}")
            print(f"  Val F1 has not improved for {early_stopping.patience} epochs")
            break

    # Save training history
    with open(save_path / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)
    print(f"Best F1-score: {best_f1:.4f} at epoch {best_epoch}")
    print(f"Model saved to: {save_path / 'best_model.pth'}")

    return history, best_f1

# MAIN
# Configuration
SPLIT_FILE = 'data/dataset_split.json' # JSON file containing dataset split and clip assigments
BATCH_SIZE = 32 # Batch size for model training
NUM_LAYERS = 2 # LSTM layers
HIDDEN_SIZE = 128 # Size of hidden states and cell states
INPUT_SIZE = 51 # 17 keypoints x 3 coords
DROPOUT = 0.4 # Percentage of values to be randomly set to 0 in each hidden state between layers 
CLASS_WEIGHTS = [1.0, 1.3] # Class weighting for model prioritisation
NUM_EPOCHS = 75 # Max number of epochs to train for
PATIENCE = 15 # Early stopping patience
MIN_DELTA = 0.001 # Minimum change to reset early stopping
LEARNING_RATE = 0.001 # Initial learning rate value for the adam optimiser for improving parameters - this gets adapted individually per parameter
WEIGHT_DECAY = 5e-4 # Penalty given to all parameters to prevent them from growing too large, reduces overfitting


print("=" * 70)
print("FALL DETECTION - LSTM MODEL - v11")
print("=" * 70)

# Create dataloaders
print("\nLoading data...")
train_loader, val_loader, test_loader = create_dataloaders(
    split_file=SPLIT_FILE,
    batch_size=BATCH_SIZE,
    num_workers=4,
    normalise=True # Normalise keypoints
)

# Create model
print("\nInitialising model...")
model = FallDetectionLSTM(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
)

# Count parameters in all layers
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Train model
history, best_f1 = train_model(
    model, train_loader, val_loader,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    class_weights=CLASS_WEIGHTS,
    save_dir='models/lstm_v11'
)

# Load best model and evaluate on validation set
print("\n" + "=" * 70)
print("EVALUATING ON VALIDATION SET")
print("=" * 70)

checkpoint = torch.load('models/lstm_v11/best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Evaluate
accuracy, precision, recall, f1, cm, preds, labels = evaluate(
    model, val_loader, device
)

print(f"\nValidation Results:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  {cm}")
print(f"\n  TN: {cm[0,0]}, FP: {cm[0,1]}")
print(f"  FN: {cm[1,0]}, TP: {cm[1,1]}")

# Save validation results
validation_results = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'confusion_matrix': cm.tolist()
}

with open('models/lstm_v11/validation_results.json', 'w') as f:
    json.dump(validation_results, f, indent=2)

print(f"\n! Validation results saved to: models/lstm_v11/validation_results.json")